In [1]:
!pip install transformers ffmpeg-python librosa noisereduce ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 994.0/994.0 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 80.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [8]:
!pip install protobuf==4.25.6 mediapipe grpcio-status --force-reinstall

  Using cached protobuf-4.25.6-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
  Using cached mediapipe-0.10.21-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (9.7 kB)
  Using cached grpcio_status-1.71.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached sounddevice-0.5.1-py3-none-any.whl.metadata (1.4 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [17]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use GPU 0 explicitly

import tensorflow as tf
print("Available GPUs:", tf.config.list_physical_devices('GPU'))

Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


C:\Users\izall\anaconda3\envs\gpu-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
import cv2
import mediapipe as mp
import os
import numpy as np
import pandas as pd
import glob
from transformers import ViTFeatureExtractor, ViTModel, Wav2Vec2FeatureExtractor, Wav2Vec2Model
import torch
import ffmpeg
import librosa
import noisereduce as nr
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

# Set environment variable for GPU usage
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {torch.cuda.get_device_name(device)}")

# Initialize MediaPipe Face Mesh
mp_face_mesh = mp.solutions.face_mesh

# Initialize ViT and Wav2Vec2 models on GPU
feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224-in21k')
vit_model = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k').to(device)  # Move to GPU

wav2vec_model_name = "facebook/wav2vec2-xls-r-300m"
wav2vec_feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(wav2vec_model_name)
wav2vec_model = Wav2Vec2Model.from_pretrained(wav2vec_model_name).to(device)  # Move to GPU

# Download and load YOLOv8-Face-Detection model
model_path = hf_hub_download(repo_id="arnabdhar/YOLOv8-Face-Detection", filename="model.pt")
yolo_face_model = YOLO(model_path)

def process_and_extract_features(frame):
    """Process frame and extract features with MediaPipe first, then YOLO-Face fallback."""
    try:
        # Initialize MediaPipe Face Mesh
        with mp_face_mesh.FaceMesh(
            static_image_mode=True,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.3,
            min_tracking_confidence=0.3
        ) as face_mesh:

            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h, w, _ = frame.shape

            # Attempt face detection with MediaPipe
            results = face_mesh.process(image_rgb)
            if results.multi_face_landmarks:
                # If face detected, process with ViT on GPU
                inputs = feature_extractor(images=image_rgb, return_tensors="pt").to(device)
                outputs = vit_model(**inputs)
                features = outputs.last_hidden_state.mean(dim=1).detach().cpu().numpy()
                return features.flatten()

            # Fallback to YOLOv8-Face if MediaPipe fails
            print("MediaPipe failed. Trying YOLOv8-Face for detection...")
            results = yolo_face_model.predict(frame, conf=0.5, verbose=False)
            if results and hasattr(results[0], 'boxes') and results[0].boxes is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                if len(boxes) > 0:
                    x1, y1, x2, y2 = boxes[0].astype(int)
                    cropped_face = frame[y1:y2, x1:x2]
                    if cropped_face.size > 0:
                        cropped_rgb = cv2.cvtColor(cropped_face, cv2.COLOR_BGR2RGB)
                        inputs = feature_extractor(images=cropped_rgb, return_tensors="pt").to(device)
                        outputs = vit_model(**inputs)
                        features = outputs.last_hidden_state.mean(dim=1).detach().cpu().numpy()
                        return features.flatten()
            print("Face detection failed for this frame.")
            return None

    except Exception as e:
        print(f"Error processing frame: {str(e)}")
        return None

def extract_audio(video_path, output_audio_path="temp_audio.wav"):
    """Extract audio from video using FFmpeg."""
    (
        ffmpeg.input(video_path)
        .output(output_audio_path, ac=1, ar=16000)
        .overwrite_output()
        .run(quiet=True)
    )
    return output_audio_path

def get_xlsr_embeddings(audio_path):
    """Extract XLS-R embeddings from audio."""
    audio, sr = librosa.load(audio_path, sr=16000)
    audio = librosa.util.normalize(audio)
    audio = nr.reduce_noise(y=audio, sr=sr)

    inputs = wav2vec_feature_extractor(
        audio,
        return_tensors="pt",
        sampling_rate=16000,
        padding="max_length",
        max_length=16000 * 10,
        truncation=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}  
    with torch.no_grad():
        outputs = wav2vec_model(**inputs)

    return outputs.last_hidden_state.mean(dim=1).cpu().squeeze().numpy()

def process_all_media_in_folder(folder_path):
    """Process all media files in the specified folder."""
    video_files = glob.glob(os.path.join(folder_path, '**', '*.mp4'), recursive=True)
    audio_data = []
    frame_data = []

    for video_path in video_files:
        print(f"Processing video: {video_path}")
        video_name = os.path.basename(video_path)

        # Process audio
        audio_path = extract_audio(video_path)
        audio_features = get_xlsr_embeddings(audio_path)
        audio_data.append({'Video File': video_name, 'Features': audio_features.tolist()})
        os.remove(audio_path)

        # Process video frames
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        
        # Ensure fps is not zero to avoid ZeroDivisionError
        if fps == 0:
            print(f"Skipping file due to zero FPS: {video_path}")
            cap.release()
            continue

        frame_interval = int(fps / 24)  # Process every 24th frame
        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_count % frame_interval == 0:
                features = process_and_extract_features(frame)
                if features is not None:
                    frame_data.append({
                        'Video File': video_name,
                        'Frame Index': frame_count,
                        'Features': features.tolist()
                    })
            frame_count += 1
        cap.release()

    # Save audio features
    audio_df = pd.DataFrame(audio_data)
    audio_df.to_csv('audio_features.csv', index=False)
    print("Audio features saved to audio_features.csv")

    # Save frame features
    frame_df = pd.DataFrame(frame_data)
    frame_df.to_csv('frame_features.csv', index=False)
    print("Frame features saved to frame_features.csv")

# Run the pipeline on your folder
folder_path = "PolyGlotFake/train"
process_all_media_in_folder(folder_path)

Using device: NVIDIA GeForce RTX 4060 Laptop GPU
Processing video: PolyGlotFake/train\fake\ar_10_to_en_MicroTts.mp4
Processing video: PolyGlotFake/train\fake\ar_10_to_en_Xtts.mp4
Processing video: PolyGlotFake/train\fake\ar_10_to_es_MicroTts.mp4
Processing video: PolyGlotFake/train\fake\ar_10_to_es_Xtts.mp4
Processing video: PolyGlotFake/train\fake\ar_10_to_fr_MicroTts.mp4
Processing video: PolyGlotFake/train\fake\ar_10_to_ja_MicroTts.mp4
Processing video: PolyGlotFake/train\fake\ar_10_to_ru_Xtts.mp4
Processing video: PolyGlotFake/train\fake\ar_10_to_zh_MicroTts.mp4
Processing video: PolyGlotFake/train\fake\ar_11_to_en_MicroTts.mp4
Processing video: PolyGlotFake/train\fake\ar_11_to_es_MicroTts.mp4
Processing video: PolyGlotFake/train\fake\ar_11_to_es_Xtts.mp4
Processing video: PolyGlotFake/train\fake\ar_11_to_fr_MicroTts.mp4
Processing video: PolyGlotFake/train\fake\ar_11_to_fr_Xtts.mp4
Processing video: PolyGlotFake/train\fake\ar_11_to_ru_MicroTts.mp4
Processing video: PolyGlotFake/tr

In [ ]:
import os
import cv2
import glob
import ffmpeg
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
import noisereduce as nr
import mediapipe as mp
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model

# Allow GPU memory growth for TensorFlow
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        for device in physical_devices:
            tf.config.experimental.set_memory_growth(device, True)
        print("GPU memory growth allowed for TensorFlow.")
    except RuntimeError as e:
        print(f"Error setting memory growth: {e}")

# Set PyTorch device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using PyTorch device: {torch.cuda.get_device_name(device)}")

# Load the VGG16 model pre-trained on ImageNet and move to GPU
base_model = VGG16(weights='imagenet')
model = Model(inputs=base_model.input, outputs=base_model.get_layer('fc1').output)

# Download and load YOLOv8-Face-Detection model
model_path = hf_hub_download(repo_id="arnabdhar/YOLOv8-Face-Detection", filename="model.pt")
yolo_face_model = YOLO(model_path)

def process_and_extract_features(frame):
    """Process frame and extract features using MediaPipe first, then YOLOv8-Face fallback"""
    try:
        # Initialize MediaPipe Face Mesh for this frame
        with mp.solutions.face_mesh.FaceMesh(
            static_image_mode=True,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.3,
            min_tracking_confidence=0.3
        ) as face_mesh:

            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h, w, _ = frame.shape

            # First try with MediaPipe
            results = face_mesh.process(image_rgb)

            if results.multi_face_landmarks:
                for face_landmarks in results.multi_face_landmarks:
                    # Extract bounding box from landmarks
                    x_min = int(min([lm.x for lm in face_landmarks.landmark]) * w)
                    y_min = int(min([lm.y for lm in face_landmarks.landmark]) * h)
                    x_max = int(max([lm.x for lm in face_landmarks.landmark]) * w)
                    y_max = int(max([lm.y for lm in face_landmarks.landmark]) * h)

                    # Extract the nose tip and lower face region
                    nose_tip = face_landmarks.landmark[1]
                    nose_tip_y = int(nose_tip.y * h)
                    lower_face_crop = frame[nose_tip_y:y_max, x_min:x_max]

                    if lower_face_crop.size > 0:
                        lower_face_crop = cv2.resize(lower_face_crop, (224, 224))
                        lower_face_crop = np.expand_dims(lower_face_crop, axis=0)
                        lower_face_crop = preprocess_input(lower_face_crop)
                        features = model.predict(lower_face_crop)  # TensorFlow uses GPU automatically
                        return features.flatten()

            # If MediaPipe failed, try YOLOv8-Face
            print("MediaPipe failed. Trying YOLOv8-Face for detection...")
            results = yolo_face_model.predict(frame, conf=0.5, verbose=False)

            if results and hasattr(results[0], 'boxes') and results[0].boxes is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()

                if len(boxes) > 0:
                    # Extract the first detected face box
                    x1, y1, x2, y2 = boxes[0].astype(int)
                    cropped_face = frame[y1:y2, x1:x2]

                    if cropped_face.size > 0:
                        # Try MediaPipe again on the cropped face
                        cropped_rgb = cv2.cvtColor(cropped_face, cv2.COLOR_BGR2RGB)
                        results_cropped = face_mesh.process(cropped_rgb)

                        if results_cropped.multi_face_landmarks:
                            ch, cw, _ = cropped_face.shape
                            for face_landmarks in results_cropped.multi_face_landmarks:
                                x_min = int(min([lm.x for lm in face_landmarks.landmark]) * cw)
                                x_max = int(max([lm.x for lm in face_landmarks.landmark]) * cw)
                                y_max = int(max([lm.y for lm in face_landmarks.landmark]) * ch)
                                nose_tip = face_landmarks.landmark[1]
                                nose_tip_y = int(nose_tip.y * ch)

                                lower_face_crop = cropped_face[nose_tip_y:y_max, x_min:x_max]
                                if lower_face_crop.size > 0:
                                    lower_face_crop = cv2.resize(lower_face_crop, (224, 224))
                                    lower_face_crop = np.expand_dims(lower_face_crop, axis=0)
                                    lower_face_crop = preprocess_input(lower_face_crop)
                                    features = model.predict(lower_face_crop)  
                                    return features.flatten()
                            print("Retry with cropped face also failed.")
            else:
                print("YOLOv8-Face could not detect a face.")

    except Exception as e:
        print(f"Error processing frame: {str(e)}")

    return None

def extract_audio(video_path, output_audio_path="temp_audio.wav"):
    ffmpeg.input(video_path).output(output_audio_path, ac=1, ar=16000).overwrite_output().run(quiet=True)
    return output_audio_path

def get_xlsr_embeddings(audio_path):
    model_name = "facebook/wav2vec2-xls-r-300m"
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
    model = Wav2Vec2Model.from_pretrained(model_name).to(device)  # Move model to GPU

    audio, sr = librosa.load(audio_path, sr=16000)
    audio = librosa.util.normalize(audio)
    audio = nr.reduce_noise(y=audio, sr=sr)

    inputs = feature_extractor(
        audio,
        return_tensors="pt",
        sampling_rate=16000,
        padding="max_length",
        max_length=16000 * 10,
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}  # Move inputs to GPU

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state.mean(dim=1).cpu().squeeze().numpy()

def process_all_media_in_folder(folder_path):
    """Process all media files in the specified folder."""
    video_files = glob.glob(os.path.join(folder_path, '**', '*.mp4'), recursive=True)

    audio_data = []
    frame_data = []

    for video_path in video_files:
        print(f"Processing video: {video_path}")
        video_name = os.path.basename(video_path)

        # Extract and process audio
        audio_path = extract_audio(video_path)
        audio_features = get_xlsr_embeddings(audio_path)
        audio_data.append({'Video File': video_name, 'Features': audio_features.tolist()})
        os.remove(audio_path)

        # Process video frames
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)

        # Ensure fps is not zero to avoid ZeroDivisionError
        if fps == 0:
            print(f"Skipping file due to zero FPS: {video_path}")
            cap.release()
            continue

        frame_interval = int(fps / 24)  # Process every 24th frame
        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_count % frame_interval == 0:
                features = process_and_extract_features(frame)
                if features is not None:
                    frame_data.append({
                        'Video File': video_name,
                        'Frame Index': frame_count,
                        'Features': features.tolist()
                    })
            frame_count += 1
        cap.release()

    # Save audio features
    audio_df = pd.DataFrame(audio_data)
    audio_df.to_csv('audio_features.csv', index=False)
    print("Audio features saved to audio_features.csv")

    # Save frame features
    frame_df = pd.DataFrame(frame_data)
    frame_df.to_csv('frame_features.csv', index=False)
    print("Frame features saved to frame_features.csv")

# Run on your folder
folder_path = "/content/drive/MyDrive/data/testyolo"
process_all_media_in_folder(folder_path)